# Logos Pretraining — Colab (Blackwell 96 GB)

End-to-end pretraining of a single **Logos** transformer with every model and trainer feature turned on:

**Architecture (Logos)**
- **KDA + Snapshot Retrieval** sub-quadratic attention with **MLA-compressed snapshots**
- **Local Sliding-Window Attention** every 4 layers (Samba-style)
- **Block Attention Residuals** (depth-wise softmax over completed blocks)
- **Three-section** Entry → Body (looped 3×) → Exit
- **Mixture-of-Experts** body: 2 shared + 32 sparse, top-4, with **cross-loop expert diversity**
- **Attention sink**, **QK norm**, **partial RoPE**

**Training pipeline**
- **Muon** (2D Linears) + **AdamW** (1D scalars + embedding + MoE router) split with differentiated base lrs and warmups (router gets a smaller lr and a longer warmup so bias-balance can stabilize before router weights start moving aggressively)
- **WSD** schedule (warmup → stable → linear decay), Muon-only **momentum ramp** + **WD anneal**
- **EMA disabled** by default to reduce memory and validation overhead
- **Rolling-N** checkpoints (best/final preserved)
- **bf16** + `torch.compile`
- **8192-token** context, **streaming** FineWeb-Edu corpus

This notebook is a thin wrapper: it builds an `argparse.Namespace` of training flags and hands it to `scripts.train.main(args)`. There's no duplicated training loop — every option (Muon split, WSD scheduler, rolling-N checkpoints, MoE bias updates) lives in `train.py` and is exercised through that single entry point. Future improvements to `train.py` show up here automatically.

> **Prerequisites:** make sure your fork of `Logos` on GitHub has the latest master branch. The clone cell below does `git fetch && git reset --hard origin/master` so subsequent runs pick up new commits — but the upstream needs to be current.


## 1. GPU sanity check

In [ ]:
!nvidia-smi

## 2. Install dependencies

Logos needs PyTorch ≥ 2.10 (for `torch.optim.Muon` and Blackwell `sm_100` kernels). If Colab's preinstalled torch is older, the next cell upgrades from the CUDA-13.0 wheel index.

In [ ]:
import os, multiprocessing
os.environ.setdefault('TORCHINDUCTOR_COMPILE_THREADS', str(multiprocessing.cpu_count()))
print(f"TORCHINDUCTOR_COMPILE_THREADS = {os.environ['TORCHINDUCTOR_COMPILE_THREADS']}")

os.environ.setdefault('TORCHINDUCTOR_PERSISTENT_REDUCTIONS', '0')
print(f"TORCHINDUCTOR_PERSISTENT_REDUCTIONS = {os.environ['TORCHINDUCTOR_PERSISTENT_REDUCTIONS']}")

# Upgrade torch only if needed (Muon requires 2.10+).
import torch, sys, subprocess
need = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (2, 10)
if need:
    print(f'torch {torch.__version__} is too old, upgrading...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--upgrade',
        'torch', '--index-url', 'https://download.pytorch.org/whl/cu130',
    ])
    print('Restart the runtime now (Runtime → Restart) and re-run from cell 1.')
else:
    print(f'torch {torch.__version__} OK.')

import torch._inductor.config as _ic
_ic.triton.persistent_reductions = False
print('inductor.triton.persistent_reductions =',
      _ic.triton.persistent_reductions)

In [ ]:
!pip install -q -U datasets transformers tiktoken einops tqdm

## 3. Get the Logos repo

Replace the URL with your fork if you have one. The cell does a `git fetch && git reset --hard origin/master` if the path already exists, so re-running picks up new commits.

In [ ]:
%cd /content/
import os, pathlib, subprocess
REPO_URL  = 'https://github.com/Rorical/Logos.git'
REPO_PATH = '/content/Logos'

if pathlib.Path(REPO_PATH).exists():
    print('Repo exists - fetching + hard-resetting to origin/master')
    subprocess.check_call(['git', '-C', REPO_PATH, 'fetch', '--all', '--prune'])
    subprocess.check_call(['git', '-C', REPO_PATH, 'reset', '--hard', 'origin/master'])
else:
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_PATH])

print('HEAD:')
subprocess.check_call(['git', '-C', REPO_PATH, 'log', '--oneline', '-1'])
%cd /content/Logos


## 4. Mount Google Drive (optional but recommended)

Checkpoints land in `MyDrive/logos-pretraining/` so they survive runtime restarts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, tarfile

CHECKPOINT_DIR = '/content/drive/MyDrive/logos-pretraining'
pathlib.Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
print('Checkpoints will go to', CHECKPOINT_DIR)

INDUCTOR_LIVE = '/content/inductor_cache'
TRITON_LIVE   = '/content/triton_cache'
pathlib.Path(INDUCTOR_LIVE).mkdir(parents=True, exist_ok=True)
pathlib.Path(TRITON_LIVE).mkdir(parents=True, exist_ok=True)
os.environ['TORCHINDUCTOR_CACHE_DIR'] = INDUCTOR_LIVE
os.environ['TRITON_CACHE_DIR'] = TRITON_LIVE

print('Inductor cache (live):', INDUCTOR_LIVE)
print('Triton cache (live)  :', TRITON_LIVE)

## 5. Imports + GPU info

In [ ]:
import sys, torch
sys.path.insert(0, '/content/Logos')

from scripts.train import build_arg_parser, main

assert torch.cuda.is_available(), 'No GPU detected — switch the Colab runtime to a Blackwell GPU.'
props = torch.cuda.get_device_properties(0)
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'Device   : {props.name}')
print(f'Memory   : {props.total_memory / 1e9:.1f} GB')
print(f'Capability: sm_{props.major}{props.minor}')


## 6. API compatibility check

Fail fast with an actionable message if the cloned repo is older than the Muon-WSD / streaming work.

In [ ]:
from models.logos import LogosConfig
import scripts.train as st

required = []
for name in ('build_arg_parser', 'main', 'PackedStream',
             'build_streaming_loaders', 'run_step_training',
             'split_param_groups', 'build_optimizer_and_scheduler'):
    if not hasattr(st, name):
        required.append(f'scripts.train.{name}')

if required:
    raise RuntimeError(
        'Cloned repo is missing required APIs: ' + ', '.join(required) +
        '\nRe-run the clone cell to fetch latest origin/master.'
    )
print('API compatibility OK.')


## Scale & memory budget

**Compute-optimal target (Chinchilla, 10B tokens).** With `D = 20·N_active`, the optimum sits at ~500M active params. The configured model is 1.30B total / 379M active — undertrained-by-Chinchilla on purpose so it fits in memory without bumping `d_model` (which would force re-tuning attention head dims). Smaller-than-optimum is fine; we trade a bit of asymptotic loss for a cleaner architecture and faster wall-clock.

**Training horizon.** `--total-tokens 10B` derives `total_steps` automatically from `world_size × batch_size × max_length` — at `bs=7 × seq=4096` on a single GPU that's ~349k steps for one pass over 10B tokens. The token budget stays fixed if you sweep batch size or move to multi-GPU, so you don't have to rescale `--total-steps` by hand. Don't run multiple epochs at compute-optimal — repeat-data returns saturate around 4 epochs and 10BT is sized for a single pass. If you want to train longer, switch the dataset to `sample-100BT` and bump the budget.

**Memory levers (96 GB Blackwell).** Current run sits at ~28 GB with `bs=1`. The headroom is being spent on:
* `--batch-size 7` + `--gradient-checkpointing --ckpt-granularity per-block` — 7× tokens/step over an un-checkpointed bs=2 baseline. Per-block ckpt recomputes one body block at a time on backward, keeping the activation footprint proportional to a single block instead of the whole stack. Expect ~70-85 GB resident on a 96 GB card. If you have memory headroom, `--ckpt-granularity per-loop` saves fewer HOP boundaries (better torch.compile fusion) but the backward recompute holds `num_body_layers` blocks of activations at once, so peak memory rises sharply — only switch when you've measured the headroom.
* `--num-body-layers 8` (was 6) and `--expert-d-ff 768` (was 512) — together push total to 1.30B / active to 379M. Optimizer state grows ~1.6× with total params.
* `--top-k 4` for body MoEs (loop-shared so the bias balancer converges fast). Boundary stacks override to `--entry-top-k 8 --exit-top-k 8` because they only see one forward per step and were saturating expert capacity at top_k=4 (load_max held at the cap_factor/num_experts ceiling).
* `--lm-head-chunk-size 1024` — engages the chunked tied-CE custom_op. With vocab=100,277 and seq=4096 the standard CE would materialise ~11 GB of fp32 logits per forward; the chunked path keeps that bounded by `chunk × V`, recomputing one logits chunk at a time on backward.

**`torch.compile` speed.** Inductor's Triton kernel compile is single-threaded by default, so a fresh compile of Logos (entry + body + exit, with the body unrolled `num_loops` times) can take many minutes pinned to one CPU core. Cell 4 sets `TORCHINDUCTOR_COMPILE_THREADS` to the host's core count, and cell 9 points `TORCHINDUCTOR_CACHE_DIR` at Drive so subsequent runs hit the cache instead of recompiling.

**If you OOM:** drop `--batch-size` to 4 first (the 7→4 drop alone cuts activation memory ~40%), then `--max-length` to 2048, then `--expert-d-ff` back to 512. Note that `--total-tokens` keeps the budget fixed, so dropping batch size only adds steps, not training data.


## 7. Build the training Namespace

This is the single source of configuration for the run. Everything below is a flag from `scripts/train.py`'s argument parser — no model build / optimizer / loop code lives in this notebook.

In [ ]:
parser = build_arg_parser()

# Single-GPU Colab run. Every CLI knob from scripts/train.py that
# applies to one card is set explicitly below — defaults included —
# so this cell doubles as a self-documenting reference. DDP-only
# flags (--ddp-find-unused-parameters, --ddp-static-graph) are
# omitted because they are no-ops without torchrun.
cli = [
    '--model', 'logos',

    # ---- Streaming corpus ----
    '--streaming',
    '--dataset',         'HuggingFaceFW/fineweb-edu',
    '--dataset-config',  'sample-10BT',
    '--text-column',     'text',
    '--val-docs',        '200',
    '--tiktoken-encoding', 'cl100k_base',  # cl100k_base | o200k_base

    # ---- Training horizon ----
    '--total-tokens', '10B',     # token budget; total_steps derived as ceil(10e9 / (ws * bs * seq))
    '--batch-size',   '8',       # 96 GB headroom + gradient checkpointing fits this comfortably
    '--max-length',   '4096',    # halve context: cuts activation memory ~50%
    '--log-every',    '50',
    '--eval-every',   '10000',
    '--save-every',   '5000',   # save before the first eval boundary so an eval crash can't wipe out hours of progress
    '--sample-every', '20000',
    '--keep-last-n',  '5',

    # ---- DataLoader (per-rank prefetch pipeline) ----
    '--num-workers',     '8',  # tokeniser-on-the-fly runs off the main thread
    '--prefetch-factor', '8',  # batches each worker keeps queued ahead

    # ---- Mixed precision + compile ----
    '--bf16',
    '--compile',
    '--compile-mode', 'max-autotune',  # default | reduce-overhead | max-autotune
    '--gradient-checkpointing',   # recompute body activations on backward; trades wall time for ~3-5x lower activation memory
    '--ckpt-granularity', 'per-block',  # 'per-block' = lowest backward peak (allows largest --batch-size); 'per-loop' = better compile fusion when memory has headroom
    '--block-residual-isolate-softmax',  # route depth-softmax+sum through an opaque custom_op so torch.compile can't fuse softmax_backward with the upstream stack/RMSNorm chain — required on sm_120 (RTX PRO 6000 Blackwell, ~99 KB SMEM/SM) where the fused backward kernel asks >128 KB and fails to compile

    # ---- Logos architecture ----
    '--d-model',          '1024',
    '--num-heads',        '16',
    '--head-dim',         '64',
    '--d-ff',             '2730',
    '--num-entry-layers', '2',
    '--num-body-layers',  '6',
    '--num-exit-layers',  '2',
    '--num-loops',        '3',
    '--dropout',          '0.0',
    '--norm-eps',         '1e-6',

    # KDA + retrieval
    '--chunk-size',          '128',
    '--conv-size',           '4',
    '--snapshot-interval',   '512',
    '--snapshot-latent-dim', '128',
    '--mem-top-k',           '16',
    '--mem-head-dim',        '64',

    # RoPE (extends context only when scaling != none)
    '--rope-scaling-type',   'none',  # none | yarn | linear | dynamic
    '--rope-scaling-factor', '1.0',
    '--yarn-beta-fast',      '32.0',
    '--yarn-beta-slow',      '1.0',
    # '--rope-original-max-position', '4096',  # required for yarn/linear/dynamic

    # SWA
    '--swa-window', '256',
    '--swa-every',  '4',
    '--swa-offset', '3',

    # MoE
    '--use-moe',
    '--num-shared-experts',   '2',
    '--num-sparse-experts',   '32',
    '--top-k',                '4',
    '--entry-top-k',          '8',  # boundary stacks see less effective routing data than the loop-shared body; raise top_k to lower per-expert load (cap = N * top_k * cap_factor / num_experts saturates the wandb 'load_max' metric at cap_factor/num_experts = 0.0625 here)
    '--exit-top-k',           '8',
    '--expert-d-ff',          '768',
    '--moe-diversity-factor', '0',
    '--bias-update-rate',     '0.02',
    '--moe-log-every',        '1000',  # 0 disables MoE expert-load wandb histograms
    # Memory-efficient LM head: chunked tied-CE. With cl100k vocab
    # (100,277) and seq=4096, the standard CE would materialise
    # B*T*V*4 ~= 11.5 GB of fp32 logits per micro-batch. The chunked
    # path streams logits in (1024 rows × V) tiles to avoid that
    # allocation, recomputing one logits chunk at a time on backward.
    '--lm-head-chunk-size', '4096',
    '--router-lr',           '1e-3',  # AdamW router group (MoE Router.linear.weight); old 4e-4 left router essentially frozen during warmup and the bias balancer oscillated load between experts
    '--router-warmup-steps', '3500',  # = --warmup-steps; old 3x stretch (10500) froze the router so long that the bias balancer was the only force shaping load — and on boundary stacks it oscillated instead of converging

    # ---- Optimizer (Muon + AdamW) ----
    '--muon',                            # --no-muon falls back to AdamW for everything
    '--muon-schedule-hyperparams',       # --no-... holds momentum/wd at start values
    '--lr',           '0.004',  # AdamW default group (norms/biases/conv)
    '--embed-lr',     '0.1',    # AdamW embedding group (token_emb / lm_head tied)
    '--muon-lr',      '0.01',   # Muon (2D Linear weights) — raw scaling
    '--weight-decay', '0.1',
    '--grad-clip',    '1.0',

    # ---- WSD schedule + Muon hyperparam ramp ----
    '--scheduler',    'wsd',     # wsd | cosine | linear | constant
    '--warmup-steps', '3500',    # ~1% of total_steps (~349k at bs=7, seq=4096, 10B tokens)
    '--decay-steps',  '70000',   # ~20% WSD cooldown (overrides --decay-frac)
    '--decay-frac',   '0.2',     # used iff --decay-steps unset
    '--muon-momentum-start',     '0.85',
    '--muon-momentum-mid',       '0.90',
    '--muon-momentum-end',       '0.95',
    '--muon-momentum-warmup-1',  '150',
    '--muon-momentum-warmup-2',  '300',
    '--muon-wd-start',           '0.2',
    '--muon-wd-end',             '0.0',

    # ---- EMA shadow weights ----
    '--ema-decay', '0.0',  # disabled by default; set 0.999 or 0.9999 to enable

    # ---- Sampling ----
    '--sample-prompt',      'In a recent study, researchers found that',
    '--sample-max-tokens',  '80',
    '--sample-temperature', '0.8',

    # ---- Output ----
    '--save-dir', CHECKPOINT_DIR,
    '--seed',     '42',

    # ---- Weights & Biases (uncomment to enable; requires `wandb login`) ----
    '--wandb',
    '--wandb-project',  'logos-pretrain',
    # '--wandb-run-name', 'logos-1.3B-fineweb10BT-colab',
    # '--wandb-tags',     'logos', 'fineweb-edu', 'colab',
    '--wandb-mode',     'online',  # online | offline | disabled
]

args = parser.parse_args(cli)
print('Run config:')
for k, v in sorted(vars(args).items()):
    print(f'  {k:<28} {v}')

## 8. Train

Single call. `main(args)` does:
1. Build tokenizer + streaming dataloaders (`build_streaming_loaders`).
2. Build the Logos model (with all features off the args).
3. Four-way Muon/AdamW split (`split_param_groups`: muon / embed / default / router) + WSD scheduler + Muon momentum/wd ramp. Router weights run on AdamW with their own (smaller) lr and longer warmup.
4. EMA is disabled by default (`--ema-decay 0.0`) to reduce memory and validation overhead.
5. `torch.compile` the forward.
6. Step-bounded loop (`run_step_training`): forward → backward → opt → sched → muon_hp → MoE bias update, with periodic eval / sample / checkpoint, rolling-N pruning.
7. Final checkpoint + `history.json` written to `--save-dir`.

In [ ]:
main(args)

## 9. (Optional) Sample from the final model after training

Loads the final checkpoint and generates a continuation. If you later enable EMA, this cell will use the EMA weights when present; otherwise it uses the live final weights.

In [ ]:
import torch
from pathlib import Path
from models.logos import LogosConfig, LogosTransformer
from utils.tokenizer import TiktokenTokenizer
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn

ckpt = torch.load(Path(CHECKPOINT_DIR) / 'checkpoint_final.pt',
                  map_location='cuda', weights_only=False)
config_dict = ckpt['model_state_dict']  # state_dict, not config

# Rebuild config from the args we passed (we still have `args` from cell 7).
cfg_kwargs = {k: v for k, v in vars(args).items() if k in LogosConfig.__dataclass_fields__}
cfg = LogosConfig(**{**cfg_kwargs, 'vocab_size': TiktokenTokenizer(args.tiktoken_encoding).vocab_size})

model = LogosTransformer(cfg).to('cuda')
model.load_state_dict(ckpt['model_state_dict'])

if ckpt.get('ema_state_dict'):
    ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(args.ema_decay), use_buffers=True).to('cuda')
    ema.load_state_dict(ckpt['ema_state_dict'])
    target = ema.module
    print('Using EMA weights')
else:
    target = model
    print('No EMA in checkpoint — using live weights')

tok = TiktokenTokenizer(args.tiktoken_encoding)
prompt = 'The history of artificial intelligence began'
ids = torch.tensor([tok.encode(prompt)], dtype=torch.long, device='cuda')
target.eval()
with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    out = target.generate(ids, max_new_tokens=200, temperature=0.7, top_k=40)
print(tok.decode(out[0].tolist()))


## 10. (Optional) Plot the loss curve

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

with (Path(CHECKPOINT_DIR) / 'history.json').open() as f:
    hist = json.load(f)

events = hist['history']
val_pts = [(h['step'], h['val']['loss']) for h in events if h.get('val') and h['val']['loss'] != float('inf')]
ema_pts = [(h['step'], h['ema_val']['loss']) for h in events if h.get('ema_val') and h['ema_val']['loss'] != float('inf')]

plt.figure(figsize=(10, 5))
if val_pts: plt.plot(*zip(*val_pts), 'o-', label='val', linewidth=2)
if ema_pts: plt.plot(*zip(*ema_pts), 's-', label='ema val', linewidth=2)
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Logos pretraining'); plt.show()
